In [4]:
import pandas as pd
import numpy as np
from tensorflow.keras import models , layers
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.pipeline import Pipeline

In [5]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [6]:
print("Dataset shape:" , df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [9]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [10]:
df_ann = df.copy()

In [11]:
df_ann["TotalCharges"] = pd.to_numeric(
    df_ann["TotalCharges"],
    errors="coerce"
)

In [12]:
if "customerID" in df_ann.columns:
    df_ann = df_ann.drop(columns=["customerID"])

In [13]:
if df_ann["Churn"].dtype == "object":
    df_ann["Churn"] = df_ann["Churn"].map({
        "No":0,
        "Yes":1
    })

In [14]:
print(df_ann["Churn"].value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [15]:
X = df_ann.drop(columns=["Churn"])
y = df_ann["Churn"]

In [16]:
print("Feature shape:" , X.shape)
print("Target shape:" , y.shape)

Feature shape: (7043, 19)
Target shape: (7043,)


In [17]:
from sklearn.model_selection import train_test_split

In [19]:
X_train_full , X_test , y_train_full , y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
X_train , X_val , y_train , y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

In [21]:
numerical_col = (
    X_train
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

In [22]:
numerical_col

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [23]:
categorical_col = (
    X_train
    .select_dtypes(include=["object" , "category" , "bool"])
    .columns
    .tolist()
)

C:\Users\rupam\AppData\Local\Temp\ipykernel_15444\1918097607.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object" , "category" , "bool"])


In [24]:
categorical_col

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [26]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

In [28]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_col
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_col
    )
])

In [29]:
X_train_processed = preprocessor.fit_transform(X_train)

In [30]:
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [31]:
X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_val_processed = np.asarray(
    X_val_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)

In [33]:
y_train = y_train.map({
    "No":0,
    "Yes":1
})

y_val = y_val.map({
    "No":0,
    "Yes":1
})

y_test = y_test.map({
    "No":0,
    "Yes":1
})

In [34]:
y_train_array = y_train.to_numpy(dtype=np.float32)
y_val_array = y_val.to_numpy(dtype=np.float32)
y_test_array = y_test.to_numpy(dtype=np.float32)

In [35]:
input_size = X_train_processed.shape[1]

In [36]:
input_size

45

In [37]:
ann_model = models.Sequential([
    layers.Input(shape=(input_size ,)),

    layers.Dense(
        64,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        32,
        activation="relu"
    ),

    layers.Dropout(0.2),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,057 (19.75 KB)

 Trainable params: 5,057 (19.75 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [39]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [40]:
history = ann_model.fit(
    X_train_processed,
    y_train_array,

    validation_data = (
        X_val_processed,
        y_val_array
    ),

    epochs=100,
    batch_size=32,

    callbacks = [
        early_stopping
    ],
    verbose=1
)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7497 - loss: 0.5049 - val_accuracy: 0.8146 - val_loss: 0.4237
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7861 - loss: 0.4444 - val_accuracy: 0.8154 - val_loss: 0.4182
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7957 - loss: 0.4319 - val_accuracy: 0.8146 - val_loss: 0.4173
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7932 - loss: 0.4323 - val_accuracy: 0.8154 - val_loss: 0.4181
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8016 - loss: 0.4252 - val_accuracy: 0.8154 - val_loss: 0.4174
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8001 - loss: 0.4234 - val_accuracy: 0.8163 - val_loss: 0.4180
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8019 - loss: 0.4207 - val_accuracy: 0.8181 - val_loss: 0.4181
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8034 - loss: 0.4194 - val_acc

In [41]:
history.history["loss"]

[0.5049276947975159,
 0.44435521960258484,
 0.43186670541763306,
 0.4322824478149414,
 0.42517682909965515,
 0.4233981668949127,
 0.4207298159599304,
 0.41938868165016174,
 0.4190983176231384,
 0.41278761625289917,
 0.41394564509391785]